# 🏆 Nairobi OS vs Pandas: NBA Dataset Benchmarks

This notebook benchmarks **Nairobi OS** (fused analytics engine) against **Pandas** using the real NBA Player Statistics dataset. All comparisons use identical operations for a fair head-to-head test.

**Workloads tested:**
- Statistical Distillation (mean, std_dev, skewness, kurtosis)
- Pearson Correlation between key player stats
- End-to-end pipeline (ingest + crunch + correlate)


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
eoinamoore_historical_nba_data_and_player_box_scores_path = kagglehub.dataset_download('eoinamoore/historical-nba-data-and-player-box-scores')

print('Data source import complete.')

100%|██████████| 1.03G/1.03G [00:09<00:00, 113MB/s]

Extracting files...


Data source import complete.


## 🔧 Setup: Load the NBA Dataset & Start Engines

In [ ]:
!pip install nairobi_os -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.0/14.0 MB 59.3 MB/s eta 0:00:00


In [ ]:
import os
import subprocess
import stat
from pathlib import Path
import nairobi_os
import pandas as pd
import gc

# Use the path from kagglehub if available, otherwise look in standard Kaggle mount points
try:
    nba_input_dir = eoinamoore_historical_nba_data_and_player_box_scores_path
except NameError:
    # Fallback to common Kaggle mount paths
    possible_paths = [
        "/kaggle/input/historical-nba-data-and-player-box-scores",
        "/kaggle/input/eoinamoore/historical-nba-data-and-player-box-scores"
    ]
    nba_input_dir = next((p for p in possible_paths if os.path.exists(p)), possible_paths[0])

csv_files = []
for dirname, _, filenames in os.walk(nba_input_dir):
    for f in filenames:
        if f.endswith('.csv'):
            csv_files.append(os.path.join(dirname, f))

if not csv_files:
    raise ValueError(f"No CSV files found in {nba_input_dir}. Please check your Kaggle data sources.")

print(f"Found {len(csv_files)} CSV files in {nba_input_dir}:")
for f in csv_files:
    size_mb = os.path.getsize(f) / (1024 * 1024)
    print(f"  {f} ({size_mb:.1f} MB)")

# Use the largest CSV (player box scores) as our benchmark dataset
nba_dataset = max(csv_files, key=lambda f: os.path.getsize(f))
print(f"\n✅ Selected dataset: {nba_dataset}")

# Copy to working directory for consistent access
dataset_basename = os.path.basename(nba_dataset)
local_dataset = f"/kaggle/working/{dataset_basename}"
# Create working dir if it doesn't exist (for non-Kaggle envs)
os.makedirs("/kaggle/working", exist_ok=True)
!cp {nba_dataset} {local_dataset}

# Quick peek at the data
df_peek = pd.read_csv(local_dataset, low_memory=False)
print(f"\n📊 Dataset shape: {df_peek.shape}")
print(f"   Columns: {list(df_peek.columns[:15])}...")
print(f"   Sample rows:")
print(df_peek.head(3).to_string())

# Ensure memory is cleared
del df_peek
gc.collect()
print("\n🧹 Memory cleared: Inspection DataFrames dropped.")

Found 9 CSV files in /root/.cache/kagglehub/datasets/eoinamoore/historical-nba-data-and-player-box-scores/versions/478:
  /root/.cache/kagglehub/datasets/eoinamoore/historical-nba-data-and-player-box-scores/versions/478/TeamStatistics.csv (34.3 MB)
  /root/.cache/kagglehub/datasets/eoinamoore/historical-nba-data-and-player-box-scores/versions/478/TeamHistories.csv (0.0 MB)
  /root/.cache/kagglehub/datasets/eoinamoore/historical-nba-data-and-player-box-scores/versions/478/Games.csv (10.7 MB)
  /root/.cache/kagglehub/datasets/eoinamoore/historical-nba-data-and-player-box-scores/versions/478/PlayerStatistics.csv (371.5 MB)
  /root/.cache/kagglehub/datasets/eoinamoore/historical-nba-data-and-player-box-scores/versions/478/Players.csv (0.5 MB)
  /root/.cache/kagglehub/datasets/eoinamoore/historical-nba-data-and-player-box-scores/versions/478/PlayerStatisticsExtended.csv (432.0 MB)
  /root/.cache/kagglehub/datasets/eoinamoore/historical-nba-data-and-player-box-scores/versions/478/LeagueSched

In [ ]:
# Start the Nairobi OS Axum Refinery daemon
import subprocess
from pathlib import Path

print("🛠 Installing D-Bus Infrastructure...")
subprocess.run(["apt-get", "update", "-qq"], capture_output=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "dbus-x11"], capture_output=True)

print("🔌 Initializing D-Bus Session...")
dbus_out = subprocess.check_output(["dbus-launch"]).decode()
for line in dbus_out.splitlines():
    if "=" in line:
        k, v = line.split("=", 1)
        os.environ[k] = v.replace(";", "").replace("'", "").replace('"', '')

print("🔐 Granting Executable Permissions...")
bin_path = Path(nairobi_os.__file__).parent / "bin" / "nairobi-axum-refinery"
os.chmod(bin_path, 0o755)

print("🔥 Igniting the Heavy Iron (Nairobi OS)...")
try:
    nairobi_os.start_refinery()
    print("✅ AXUM REFINERY ONLINE")
except Exception as e:
    print(f"\n💥 Ignition Failed: {e}")
    os.system("cat ~/.nairobi_refinery.log 2>/dev/null || echo 'No log file'")

🛠 Installing D-Bus Infrastructure...
🔌 Initializing D-Bus Session...
🔐 Granting Executable Permissions...
🔥 Igniting the Heavy Iron (Nairobi OS)...
✅ AXUM REFINERY ONLINE


## 📊 Benchmark 1: Statistical Distillation (3 Iterations)

Compute **mean, std_dev, skewness, kurtosis** on the `points` column — the core statistical primitives. **Timeout:60seconds**

In [ ]:
import time
import json
import numpy as np
import pandas as pd
import nairobi_os
import gc
import os

NUM_ITERATIONS = 3
TIMEOUT_SEC = 60
column = "points"

# ── PANDA BENCHMARK ──────────────────────────────
print("  Pandas Statistical Distillation...")
pandas_ingest_times = []
pandas_crunch_times = []
pandas_totals = []
pandas_results = None

# Independent timer for Pandas
pandas_start_wall = time.perf_counter()

for i in range(NUM_ITERATIONS):
    if time.perf_counter() - pandas_start_wall > TIMEOUT_SEC:
        print(f"  ☐ Pandas timeout reached after {i} iterations")
        break

    t0 = time.perf_counter_ns()
    df_p = pd.read_csv(local_dataset, low_memory=False)
    ingest_ms = (time.perf_counter_ns() - t0) / 1_000_000
    pandas_ingest_times.append(ingest_ms)

    t1 = time.perf_counter_ns()
    mean_val = df_p[column].mean()
    std_val = df_p[column].std()
    skew_val = df_p[column].skew()
    kurt_val = df_p[column].kurt()
    crunch_ms = (time.perf_counter_ns() - t1) / 1_000_000
    pandas_crunch_times.append(crunch_ms)
    pandas_totals.append(ingest_ms + crunch_ms)

    if pandas_results is None:
        pandas_results = {"mean": mean_val, "std_dev": std_val, "skewness": skew_val, "kurtosis": kurt_val}

    del df_p
    gc.collect()

print(f"  Mean Latency (Total): {np.mean(pandas_totals):.2f} ms")

# ── NAIROBI BENCHMARK ───────────────────────────
print("\n★ Nairobi OS Statistical Distillation...")
nairobi_ingest_times = []
nairobi_crunch_times = []
nairobi_totals = []
nairobi_results = None

# Independent timer for Nairobi
nairobi_start_wall = time.perf_counter()

for i in range(NUM_ITERATIONS):
    if time.perf_counter() - nairobi_start_wall > TIMEOUT_SEC:
        print(f"  ☐ Nairobi timeout reached after {i} iterations")
        break

    t0 = time.perf_counter_ns()
    df_n = nairobi_os.read_csv(local_dataset)
    ingest_ms = (time.perf_counter_ns() - t0) / 1_000_000
    nairobi_ingest_times.append(ingest_ms)

    t1 = time.perf_counter_ns()
    mean_val = df_n[column].mean()
    std_val = df_n[column].std_dev()
    skew_val = df_n[column].skewness()
    kurt_val = df_n[column].kurtosis()
    crunch_ms = (time.perf_counter_ns() - t1) / 1_000_000
    nairobi_crunch_times.append(crunch_ms)
    nairobi_totals.append(ingest_ms + crunch_ms)

    if nairobi_results is None:
        nairobi_results = {"mean": mean_val, "std_dev": std_val, "skewness": skew_val, "kurtosis": kurt_val}

    df_n.free()
    gc.collect()

print(f"  Mean Latency (Total): {np.mean(nairobi_totals):.2f} ms")

# ── VALIDATION ───────────────────────────────────
diff = abs(pandas_results["mean"] - nairobi_results["mean"])
print(f"\n☑ Accuracy Validation (Mean Diff): {diff:.8f}")

# ── SUMMARY TABLE ────────────────────────────────
print("\n" + "="*60)
print("BENCHMARK SUMMARY: Statistical Distillation")
print("="*60)
print(f"{'Metric':<25} {'Pandas':<15} {'Nairobi':<15} {'Speedup':<10}")
print("-" * 60)
print(f"{'Avg Total Time (ms)':<25} {np.mean(pandas_totals):<15.2f} {np.mean(nairobi_totals):<15.2f} {np.mean(pandas_totals)/np.mean(nairobi_totals):.2f}x")
print("="*60)

gc.collect()

  Pandas Statistical Distillation...
  Mean Latency (Total): 13011.84 ms

★ Nairobi OS Statistical Distillation...
  Mean Latency (Total): 3658.72 ms

☑ Accuracy Validation (Mean Diff): 0.00000000

BENCHMARK SUMMARY: Statistical Distillation
Metric                    Pandas          Nairobi         Speedup   
------------------------------------------------------------
Avg Total Time (ms)       13011.84        3658.72         3.56x


0

## 📊 Benchmark 2: Pearson Correlation (3 Iterations)

Compute Pearson correlation between **points** and **assists** — a pairwise statistical operation.

In [ ]:
import gc
import time
import numpy as np
import pandas as pd
import json
import nairobi_os

# Define benchmark parameters locally to avoid NameErrors
NUM_ITERATIONS = 3
TIMEOUT_SEC = 60

# ── PANDA BENCHMARK ──────────────────────────────
print("☐ Pandas Pearson Correlation...")
pandas_corr_times = []
pandas_corr_result = None

# Independent timer for Pandas
pandas_loop_start = time.perf_counter()
for i in range(NUM_ITERATIONS):
    if time.perf_counter() - pandas_loop_start > TIMEOUT_SEC:
        print(f"  ☐ Pandas timeout reached after {i} iterations")
        break

    t0 = time.perf_counter_ns()
    df_p = pd.read_csv(local_dataset, low_memory=False)
    corr_val = df_p["points"].corr(df_p["assists"], method="pearson")
    elapsed_ms = (time.perf_counter_ns() - t0) / 1_000_000
    pandas_corr_times.append(elapsed_ms)

    if pandas_corr_result is None:
        pandas_corr_result = corr_val

    del df_p
    gc.collect()

print(f"  Mean Latency: {np.mean(pandas_corr_times):.2f} ms")

# ── NAIROBI BENCHMARK ───────────────────────────
print("\n★ Nairobi OS Pearson Correlation...")
nairobi_corr_times = []
nairobi_corr_result = None

# Independent timer for Nairobi
nairobi_loop_start = time.perf_counter()
for i in range(NUM_ITERATIONS):
    if time.perf_counter() - nairobi_loop_start > TIMEOUT_SEC:
        print(f"  ☐ Nairobi timeout reached after {i} iterations")
        break

    t0 = time.perf_counter_ns()
    df_n = nairobi_os.read_csv(local_dataset)
    corr_res = df_n.correlate("points,assists")
    elapsed_ms = (time.perf_counter_ns() - t0) / 1_000_000
    nairobi_corr_times.append(elapsed_ms)

    if nairobi_corr_result is None:
        nairobi_corr_result = corr_res["pearson"]

    df_n.free()
    gc.collect()

print(f"  Mean Latency: {np.mean(nairobi_corr_times):.2f} ms")

# ── VALIDATION ───────────────────────────────────
accuracy_diff = abs(pandas_corr_result - nairobi_corr_result)
print(f"\n☑ Accuracy Validation (Diff): {accuracy_diff:.8f}")

# ── SUMMARY ──────────────────────────────────────
print("\n" + "="*60)
print("BENCHMARK SUMMARY: Pearson Correlation")
print("="*60)
print(f"{'Metric':<25} {'Pandas (ms)':<15} {'Nairobi (ms)':<15} {'Speedup':<10}")
print("-" * 60)
print(f"{'Correlation Ops':<25} {np.mean(pandas_corr_times):<15.2f} {np.mean(nairobi_corr_times):<15.2f} {np.mean(pandas_corr_times)/np.mean(nairobi_corr_times):.2f}x")
print("="*60)

☐ Pandas Pearson Correlation...
  Mean Latency: 13284.52 ms

★ Nairobi OS Pearson Correlation...
  Mean Latency: 4240.89 ms

☑ Accuracy Validation (Diff): 0.02986630

BENCHMARK SUMMARY: Pearson Correlation
Metric                    Pandas (ms)     Nairobi (ms)    Speedup   
------------------------------------------------------------
Correlation Ops           13284.52        4240.89         3.13x


## 📊 Benchmark 3: Fused Pipeline (Ingest + Crunch + Correlate) 3 iterations
End-to-end fused analytics: ingest, compute statistics, and correlate in a single call via `nairobi_os.data.pipeline()`.

In [ ]:
import gc
import time
import numpy as np
import pandas as pd
import json
import nairobi_os

NUM_ITERATIONS = 3
TIMEOUT_SEC = 60

# ── FUSED PIPELINE BENCHMARK ─────────────────────
print("🦁 Nairobi OS Fused Pipeline (Ingest + Crunch + Correlate)...")
pipeline_times = []
pipeline_result = None

nairobi_loop_start = time.perf_counter()
for i in range(NUM_ITERATIONS):
    if time.perf_counter() - nairobi_loop_start > TIMEOUT_SEC:
        print(f"  ☐ Nairobi timeout reached after {i} iterations")
        break
    t0 = time.perf_counter_ns()
    result_json = nairobi_os.data.pipeline(
        local_dataset,
        "points",
        "assists,reboundsTotal"
    )
    elapsed_ms = (time.perf_counter_ns() - t0) / 1_000_000
    pipeline_times.append(elapsed_ms)
    if pipeline_result is None:
        pipeline_result = json.loads(result_json)
    gc.collect()

print(f"  Mean total time: {np.mean(pipeline_times):.2f} ms")

# ── COMPARISON: Pandas equivalent (3 separate ops) ──
print("\n🐼 Pandas Equivalent (Read + Stats + Correlate)...")
pandas_pipeline_times = []

pandas_loop_start = time.perf_counter()
for i in range(NUM_ITERATIONS):
    if time.perf_counter() - pandas_loop_start > TIMEOUT_SEC:
        print(f"  ☐ Pandas timeout reached after {i} iterations")
        break
    t0 = time.perf_counter_ns()
    df_p = pd.read_csv(local_dataset, low_memory=False)
    _ = df_p["points"].mean()
    _ = df_p["points"].std()
    _ = df_p["assists"].corr(df_p["reboundsTotal"], method="pearson")
    elapsed_ms = (time.perf_counter_ns() - t0) / 1_000_000
    pandas_pipeline_times.append(elapsed_ms)

    del df_p
    gc.collect()

print(f"  Mean total time: {np.mean(pandas_pipeline_times):.2f} ms")

# ── SUMMARY ──────────────────────────────────────
print("\n" + "="*60)
print("BENCHMARK SUMMARY: Fused Pipeline vs Pandas")
print("="*60)
pipeline_speedup = np.mean(pandas_pipeline_times) / np.mean(pipeline_times)
print(f"{'Metric':<30} {'Pandas (ms)':<15} {'Nairobi (ms)':<15} {'Speedup':<10}")
print("-" * 60)
print(f"{'Full Pipeline':<30} {np.mean(pandas_pipeline_times):<15.2f} {np.mean(pipeline_times):<15.2f} {pipeline_speedup:.2f}x")
print("="*60)

gc.collect()

🦁 Nairobi OS Fused Pipeline (Ingest + Crunch + Correlate)...
  Mean total time: 3704.19 ms

🐼 Pandas Equivalent (Read + Stats + Correlate)...
  Mean total time: 12659.81 ms

BENCHMARK SUMMARY: Fused Pipeline vs Pandas
Metric                         Pandas (ms)     Nairobi (ms)    Speedup   
------------------------------------------------------------
Full Pipeline                  12659.81        3704.19         3.42x


0

## 📈 Results Summary

Run all three benchmarks above and compare the results table below.

In [ ]:
import pandas as pd
import numpy as np

# Collect results from all benchmarks
results = {
    "Benchmark": [
        "Statistical Distillation (Ingest)",
        "Statistical Distillation (Crunch)",
        "Statistical Distillation (Total)",
        "Pearson Correlation",
        "Fused Pipeline (Nairobi)",
        "Fused Pipeline (Pandas)",
    ],
    "Pandas (ms)": [
        np.mean(pandas_ingest_times),
        np.mean(pandas_crunch_times),
        np.mean(pandas_totals),
        np.mean(pandas_corr_times),
        np.mean(pandas_pipeline_times),
        np.mean(pandas_pipeline_times),
    ],
    "Nairobi (ms)": [
        np.mean(nairobi_ingest_times),
        np.mean(nairobi_crunch_times),
        np.mean(nairobi_totals),
        np.mean(nairobi_corr_times),
        np.mean(pipeline_times),
        np.mean(pandas_pipeline_times),
    ],
}

df_results = pd.DataFrame(results)
df_results["Speedup"] = df_results["Pandas (ms)"] / df_results["Nairobi (ms)"]
df_results["Pandas (ms)"] = df_results["Pandas (ms)"].apply(lambda x: f"{x:.2f}")
df_results["Nairobi (ms)"] = df_results["Nairobi (ms)"].apply(lambda x: f"{x:.2f}")
df_results["Speedup"] = df_results["Speedup"].apply(lambda x: f"{x:.2f}x")

from IPython.display import display, HTML
display(HTML('<h3>🏆 Nairobi OS vs Pandas — Benchmark Results</h3>'))
display(df_results.to_html(index=False, table_id='bench-table'))

'<table border="1" class="dataframe" id="bench-table">\n  <thead>\n    <tr style="text-align: right;">\n      <th>Benchmark</th>\n      <th>Pandas (ms)</th>\n      <th>Nairobi (ms)</th>\n      <th>Speedup</th>\n    </tr>\n  </thead>\n  <tbody>\n    <tr>\n      <td>Statistical Distillation (Ingest)</td>\n      <td>12997.02</td>\n      <td>493.66</td>\n      <td>26.33x</td>\n    </tr>\n    <tr>\n      <td>Statistical Distillation (Crunch)</td>\n      <td>14.82</td>\n      <td>3165.06</td>\n      <td>0.00x</td>\n    </tr>\n    <tr>\n      <td>Statistical Distillation (Total)</td>\n      <td>13011.84</td>\n      <td>3658.72</td>\n      <td>3.56x</td>\n    </tr>\n    <tr>\n      <td>Pearson Correlation</td>\n      <td>13284.52</td>\n      <td>4240.89</td>\n      <td>3.13x</td>\n    </tr>\n    <tr>\n      <td>Fused Pipeline (Nairobi)</td>\n      <td>12659.81</td>\n      <td>3704.19</td>\n      <td>3.42x</td>\n    </tr>\n    <tr>\n      <td>Fused Pipeline (Pandas)</td>\n      <td>12659.81</td

## 🔧 Teardown

Stop the Axum refinery daemon when done.

In [ ]:
nairobi_os.stop_refinery()
print("🛑 Axum refinery stopped.")

🛑 Nairobi OS refinery stopped.
